In [ ]:
import json
import os
import numpy as np
import pandas as pd
import torch

from uni2ts.model.moirai_moe import MoiraiMoEForecast, MoiraiMoEModule


import time

start_time = time.time()

def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Germany", "Ireland", "Portugal","Denmark"]
#days = ["day1", "day2", "day3", "day4", "day5"]
days = ["day1"]


features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
#    "price_eur_kwh"
]



PREDICTION_LENGTH = 96
CONTEXT_LENGTH = 672   # 7 days of 15-min data
PATCH_SIZE = 16        # Moirai-MoE usually uses 16
NUM_SAMPLES = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# LOAD SPLIT DAYS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# LOAD MOIRAI-MOE BASE
# ============================================================
print(f"Loading Moirai-MoE base on {DEVICE}...")

module = MoiraiMoEModule.from_pretrained("Salesforce/moirai-moe-1.0-R-base")

model = MoiraiMoEForecast(
    module=module,
    prediction_length=PREDICTION_LENGTH,
    context_length=CONTEXT_LENGTH,
    patch_size=PATCH_SIZE,
    num_samples=NUM_SAMPLES,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)

model = model.to(DEVICE)
model.eval()

# ============================================================
# HELPER FUNCTION
# ============================================================
def forecast_one_series(model, history_values, device):
    """
    history_values: numpy array of shape (context_length,)
    returns: numpy array of shape (prediction_length,)
    """

    # Shape: (batch, time, variate) = (1, context_length, 1)
    past_target = torch.tensor(
        history_values, dtype=torch.float32, device=device
    ).view(1, -1, 1)

    # all observed
    past_observed_target = torch.ones_like(past_target, dtype=torch.bool)

    # no padding
    past_is_pad = torch.zeros(
        (1, past_target.shape[1]), dtype=torch.bool, device=device
    )

    with torch.no_grad():
        forecast = model(
            past_target=past_target,
            past_observed_target=past_observed_target,
            past_is_pad=past_is_pad,
        )

    forecast_np = forecast.detach().cpu().numpy()

    if forecast_np.ndim == 3:
        y_pred = np.median(forecast_np[0], axis=0)
    elif forecast_np.ndim == 2:
        y_pred = forecast_np[0]
    else:
        raise ValueError(f"Unexpected forecast shape: {forecast_np.shape}")

    return y_pred

# ============================================================
# MAIN LOOP
# ============================================================
rmse_results = []
for country in countries:
    print("Processing country:", country)
    country_start_time = time.time()

    data_path = rf"{DATA_DIR}\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    # --------------------------------------------------------
    # COUNTRY-SPECIFIC FEATURES
    # --------------------------------------------------------
    country_features = [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation",
        "direct_radiation",
    ]

    if country == "Denmark":
        country_features.append("price_eur_kwh")

    households = [col for col in df.columns if col not in country_features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            s_train = df.loc[df.index < cutoff, household].dropna()

            if len(s_train) < CONTEXT_LENGTH:
                print(f"      Skipping {household}: not enough context ({len(s_train)} < {CONTEXT_LENGTH})")
                continue

            history_values = s_train.iloc[-CONTEXT_LENGTH:].to_numpy()

            if np.isnan(history_values).any():
                print(f"      Skipping {household}: NaNs in history")
                continue

            y_true_series = df.loc[df.index >= cutoff, household].head(PREDICTION_LENGTH)

            if len(y_true_series) < PREDICTION_LENGTH:
                print(f"      Skipping {household}: not enough future observations")
                continue

            if y_true_series.isna().any():
                print(f"      Skipping {household}: NaNs in future truth")
                continue

            y_pred = forecast_one_series(model, history_values, DEVICE)

            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=y_true_series.index)

            predictions_df_all_households[household] = y_pred

            y_true = y_true_series.to_numpy()
            rmse = root_mean_squared_error(y_true, y_pred)
            rmse_households.append(rmse)

        if len(rmse_households) == 0:
            print(f"      No valid households for {country} - {day}")
            continue

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        output = rf"{OUT_DIR}\MoiraiMoE_base_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

    # ========================================================
    # COUNTRY RUNTIME
    # ========================================================
    country_runtime = time.time() - country_start_time

    print(
        f"\nTotal runtime for {country}: "
        f"{country_runtime:.2f} seconds"
    )

    # ========================================================
    # SAVE / UPDATE JSON WITHOUT OVERWRITING EXISTING CONTENT
    # ========================================================
    json_path = os.path.join(
        OUT_DIR,
        f"time_spend_{country}.json"
    )

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            runtime_dict = json.load(f)
    else:
        runtime_dict = {}

    if "Foundational" not in runtime_dict:
        runtime_dict["Foundational"] = {}

    runtime_dict["Foundational"]["Moirai_moe_base"] = country_runtime

    with open(json_path, "w") as f:
        json.dump(runtime_dict, f, indent=4)

    print(f"Saved/updated runtime JSON: {json_path}")